# 04 - Descriptive Analytics

## Objective

This notebook computes the descriptive analytics required by the project guide — KPIs and operational comparisons covering flight delays, cancellations, airlines, airports, routes, and time periods.

Reference tables are used where coded values require descriptive labels. The analysis focuses on the most relevant findings for the predictive and operational stages of the project.

Post-departure outcome fields (e.g., departure delay) may appear in descriptive comparisons here, but are never treated as predictors — leakage-safe feature selection is enforced separately during feature engineering.


#### Load configuration and datasets

In [0]:
import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

from config import project_config as cfg
from pyspark.sql import functions as F

df = spark.read.table(cfg.CLEAN_TABLE)

df_airlines = (
    spark.read.table(cfg.AIRLINES_LOOKUP_TABLE)
    .select(
        F.col("Code").alias(cfg.AIRLINE_COLUMN),
        F.col("Description").alias("airline_name")
    )
)

df_months = (
    spark.read.table(cfg.MONTHS_LOOKUP_TABLE)
    .select(
        F.col("Code").cast("int").alias(cfg.MONTH_COLUMN),
        F.col("Description").alias("month_name")
    )
)

df_weekdays = (
    spark.read.table(cfg.WEEKDAYS_LOOKUP_TABLE)
    .select(
        F.col("Code").cast("int").alias(cfg.DAY_OF_WEEK_COLUMN),
        F.col("Description").alias("weekday_name")
    )
)

df_cancellation_codes = (
    spark.read.table(cfg.CANCELLATION_LOOKUP_TABLE)
    .select(
        F.col("Code").alias(cfg.CANCELLATION_CODE_COLUMN),
        F.col("Description").alias("cancellation_reason")
    )
)

df_airports_origin = (
    spark.read.table(cfg.AIRPORTS_LOOKUP_TABLE)
    .select(
        F.col("Code").alias(cfg.ORIGIN_COLUMN),
        F.col("Description").alias("origin_airport_name")
    )
)

df_airports_destination = (
    spark.read.table(cfg.AIRPORTS_LOOKUP_TABLE)
    .select(
        F.col("Code").alias(cfg.DESTINATION_COLUMN),
        F.col("Description").alias("destination_airport_name")
    )
)

df_operated = df.filter(
    (F.col(cfg.CANCELLED_COLUMN) == 0)
    & (F.col(cfg.DIVERTED_COLUMN) == 0)
)

print("Using cleaned dataset")

print(f"Total cleaned flights: {df.count():,}")
print(f"Operated flights: {df_operated.count():,}")

#### Helper functions

The following helper function calculates an approximate 95% confidence interval
for delay rates using the normal approximation for proportions. This provides
an indication of the statistical uncertainty associated with each reported
delay rate.

In [0]:
def add_delay_rate_confidence_interval(df):
    """
    Add an approximate 95% confidence interval for delay rates.
    """

    return (
        df
        .withColumn(
            "delay_proportion",
            F.col("delayed_flights")
            / F.col("operated_flights")
        )
        .withColumn(
            "standard_error",
            F.sqrt(
                F.col("delay_proportion")
                * (1 - F.col("delay_proportion"))
                / F.col("operated_flights")
            )
        )
        .withColumn(
            "delay_rate_ci_lower",
            F.round(
                F.greatest(
                    F.lit(0.0),
                    (
                        F.col("delay_proportion")
                        - 1.96 * F.col("standard_error")
                    ) * 100
                ),
                2
            )
        )
        .withColumn(
            "delay_rate_ci_upper",
            F.round(
                F.least(
                    F.lit(100.0),
                    (
                        F.col("delay_proportion")
                        + 1.96 * F.col("standard_error")
                    ) * 100
                ),
                2
            )
        )
        .drop(
            "delay_proportion",
            "standard_error"
        )
    )

#### Overall operational indicators

This section summarizes the main outcomes of the flight operations dataset.

In [0]:
overall_indicators = (
    df
    .agg(
        F.count("*").alias("total_flights"),

        F.sum(
            F.when(
                F.col(cfg.ARRIVAL_DELAY_FLAG_COLUMN) == 1,
                1
            ).otherwise(0)
        ).alias("delayed_flights"),

        F.sum(
            F.when(
                (F.col(cfg.CANCELLED_COLUMN) == 0)
                & (F.col(cfg.DIVERTED_COLUMN) == 0),
                1
            ).otherwise(0)
        ).alias("operated_flights"),

        F.avg(
            F.when(
                (F.col(cfg.CANCELLED_COLUMN) == 0)
                & (F.col(cfg.DIVERTED_COLUMN) == 0),
                F.col(cfg.ARRIVAL_DELAY_COLUMN)
            )
        ).alias("average_arrival_delay"),

        F.expr(
            f"""
            percentile_approx(
                CASE
                    WHEN {cfg.CANCELLED_COLUMN} = 0
                    AND {cfg.DIVERTED_COLUMN} = 0
                    THEN {cfg.ARRIVAL_DELAY_COLUMN}
                END,
                0.5
            )
            """
        ).alias("median_arrival_delay"),

        F.avg(
            F.when(
                F.col(cfg.ARRIVAL_DELAY_FLAG_COLUMN) == 1,
                F.col(cfg.ARRIVAL_DELAY_COLUMN)
            )
        ).alias("average_delay_among_delayed_flights"),

        F.sum(
            F.when(
                (F.col(cfg.CANCELLED_COLUMN) == 0)
                & (F.col(cfg.DIVERTED_COLUMN) == 0)
                & (F.col(cfg.ARRIVAL_DELAY_FLAG_COLUMN) == 0),
                1
            ).otherwise(0)
        ).alias("early_or_on_time_flights"),

        F.sum(
            F.col(cfg.CANCELLED_COLUMN)
        ).alias("cancelled_flights"),

        F.sum(
            F.col(cfg.DIVERTED_COLUMN)
        ).alias("diverted_flights")
    )
    .withColumn(
        "delay_rate",
        F.round(
            F.col("delayed_flights")
            / F.col("operated_flights")
            * 100,
            2
        )
    )
    .withColumn(
        "early_or_on_time_rate",
        F.round(
            F.col("early_or_on_time_flights")
            / F.col("operated_flights")
            * 100,
            2
        )
    )
    .withColumn(
        "cancellation_rate",
        F.round(
            F.col("cancelled_flights")
            / F.col("total_flights")
            * 100,
            2
        )
    )
    .withColumn(
        "diversion_rate",
        F.round(
            F.col("diverted_flights")
            / F.col("total_flights")
            * 100,
            2
        )
    )
    .withColumn(
        "average_arrival_delay",
        F.round(F.col("average_arrival_delay"), 2)
    )
    .withColumn(
        "median_arrival_delay",
        F.round(F.col("median_arrival_delay"), 2)
    )
    .withColumn(
        "average_delay_among_delayed_flights",
        F.round(
            F.col("average_delay_among_delayed_flights"),
            2
        )
    )
)

display(overall_indicators)

#### Monthly operational patterns

This section examines whether flight delays and cancellations vary across the months of the year.

In [0]:
monthly_performance = (
    df
    .groupBy(cfg.MONTH_COLUMN)
    .agg(
        F.count("*").alias("total_flights"),

        F.sum(
            F.when(
                F.col(cfg.ARRIVAL_DELAY_FLAG_COLUMN) == 1,
                1
            ).otherwise(0)
        ).alias("delayed_flights"),

        F.sum(
            F.col(cfg.CANCELLED_COLUMN)
        ).alias("cancelled_flights"),

        F.sum(
            F.col(cfg.DIVERTED_COLUMN)
        ).alias("diverted_flights")
    )
    .withColumn(
        "operated_flights",
        F.col("total_flights")
        - F.col("cancelled_flights")
        - F.col("diverted_flights")
    )
    .withColumn(
        "delay_rate",
        F.round(
            F.col("delayed_flights")
            / F.col("operated_flights")
            * 100,
            2
        )
    )
    .withColumn(
        "cancellation_rate",
        F.round(
            F.col("cancelled_flights")
            / F.col("total_flights")
            * 100,
            2
        )
    )
    .join(
        df_months,
        cfg.MONTH_COLUMN,
        "left"
    )
    .select(
        cfg.MONTH_COLUMN,
        "month_name",
        "total_flights",
        "operated_flights",
        "delayed_flights",
        "delay_rate",
        "cancelled_flights",
        "cancellation_rate"
    )
    .orderBy(cfg.MONTH_COLUMN)
)

display(monthly_performance)

#### Visualization

The following chart highlights the monthly variation in arrival delay rates and helps identify seasonal patterns throughout the year.

In [0]:
import matplotlib.pyplot as plt

monthly_plot = (
    monthly_performance
    .select(
        cfg.MONTH_COLUMN,
        "month_name",
        "delay_rate"
    )
    .orderBy(cfg.MONTH_COLUMN)
    .toPandas()
)

plt.figure(figsize=(10, 5))

plt.plot(
    monthly_plot["month_name"],
    monthly_plot["delay_rate"],
    marker="o"
)

plt.title("Monthly Flight Delay Rate")
plt.xlabel("Month")
plt.ylabel("Delay Rate (%)")
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.show()

#### Weekday delay patterns

This section evaluates whether arrival-delay rates vary by day of the week.

In [0]:
weekday_performance = (
    df_operated
    .groupBy(cfg.DAY_OF_WEEK_COLUMN)
    .agg(
        F.count("*").alias("operated_flights"),

        F.sum(
            F.when(
                F.col(cfg.ARRIVAL_DELAY_FLAG_COLUMN) == 1,
                1
            ).otherwise(0)
        ).alias("delayed_flights")
    )
    .withColumn(
        "delay_rate",
        F.round(
            F.col("delayed_flights")
            / F.col("operated_flights")
            * 100,
            2
        )
    )
    .join(
        df_weekdays,
        cfg.DAY_OF_WEEK_COLUMN,
        "left"
    )
    .select(
        cfg.DAY_OF_WEEK_COLUMN,
        "weekday_name",
        "operated_flights",
        "delayed_flights",
        "delay_rate"
    )
    .orderBy(cfg.DAY_OF_WEEK_COLUMN)
)

display(weekday_performance)

#### Visualization

The following chart compares the arrival delay rate across the days of the week, helping identify whether certain weekdays consistently experience higher delay rates.

In [0]:
import matplotlib.pyplot as plt

weekday_plot = (
    weekday_performance
    .select(
        cfg.DAY_OF_WEEK_COLUMN,
        "weekday_name",
        "delay_rate"
    )
    .orderBy(cfg.DAY_OF_WEEK_COLUMN)
    .toPandas()
)

plt.figure(figsize=(9, 5))

plt.bar(
    weekday_plot["weekday_name"],
    weekday_plot["delay_rate"]
)

plt.title("Arrival Delay Rate by Day of the Week")
plt.xlabel("Day of the Week")
plt.ylabel("Delay Rate (%)")
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

#### Airline performance

Airlines are compared using operated-flight volume, arrival-delay rate, and average departure and arrival delays.

Only airlines meeting the minimum operational-flight threshold defined in the project configuration are included.

In [0]:
airline_performance = (
    df_operated
    .groupBy(cfg.AIRLINE_COLUMN)
    .agg(
        F.count("*").alias("operated_flights"),

        F.sum(
            F.when(
                F.col(cfg.ARRIVAL_DELAY_FLAG_COLUMN) == 1,
                1
            ).otherwise(0)
        ).alias("delayed_flights"),

        F.avg(
            cfg.DEPARTURE_DELAY_COLUMN
        ).alias("average_departure_delay"),

        F.avg(
            cfg.ARRIVAL_DELAY_COLUMN
        ).alias("average_arrival_delay")
    )
    .filter(
        F.col("operated_flights")
        >= cfg.MIN_OPERATIONAL_FLIGHTS
    )
    .withColumn(
        "delay_rate",
        F.round(
            F.col("delayed_flights")
            / F.col("operated_flights")
            * 100,
            2
        )
    )
    .withColumn(
        "average_departure_delay",
        F.round(
            F.col("average_departure_delay"),
            2
        )
    )
    .withColumn(
        "average_arrival_delay",
        F.round(
            F.col("average_arrival_delay"),
            2
        )
    )
    .join(
        df_airlines,
        cfg.AIRLINE_COLUMN,
        "left"
    )
    .select(
        cfg.AIRLINE_COLUMN,
        "airline_name",
        "operated_flights",
        "delayed_flights",
        "delay_rate",
        "average_departure_delay",
        "average_arrival_delay"
    )
    .orderBy(
        F.col("delay_rate").desc()
    )
)

# Add approximate 95% confidence intervals
airline_performance = add_delay_rate_confidence_interval(
    airline_performance
)

display(
    airline_performance.select(
        cfg.AIRLINE_COLUMN,
        "airline_name",
        "operated_flights",
        "delayed_flights",
        "delay_rate",
        "delay_rate_ci_lower",
        "delay_rate_ci_upper",
        "average_departure_delay",
        "average_arrival_delay"
    ).limit(cfg.TOP_N_RESULTS)
)

#### Visualization

The following chart compares arrival delay rates across airlines, making it easier to identify the carriers with the highest proportion of delayed flights.

In [0]:
airline_plot = (
    airline_performance
    .select(
        "airline_name",
        "delay_rate"
    )
    .limit(cfg.TOP_N_RESULTS)
    .orderBy(F.col("delay_rate"))
    .toPandas()
)

plt.figure(figsize=(10, 6))

plt.barh(
    airline_plot["airline_name"],
    airline_plot["delay_rate"]
)

plt.title("Airline Arrival Delay Rate")
plt.xlabel("Delay Rate (%)")
plt.ylabel("Airline")
plt.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

#### Airport performance

Origin airports are compared by flight volume and arrival-delay rate. Only airports with sufficient operated-flight volume are included to avoid misleading results from very small samples.

In [0]:
origin_airport_performance = (
    df_operated
    .join(
        df_airports_origin,
        cfg.ORIGIN_COLUMN,
        "left"
    )
    .groupBy(
        cfg.ORIGIN_COLUMN,
        "origin_airport_name",
        cfg.ORIGIN_CITY_COLUMN
    )
    .agg(
        F.count("*").alias("operated_flights"),

        F.sum(
            F.when(
                F.col(cfg.ARRIVAL_DELAY_FLAG_COLUMN) == 1,
                1
            ).otherwise(0)
        ).alias("delayed_flights")
    )
    .filter(
        F.col("operated_flights")
        >= cfg.MIN_OPERATIONAL_FLIGHTS
    )
    .withColumn(
        "delay_rate",
        F.round(
            F.col("delayed_flights")
            / F.col("operated_flights")
            * 100,
            2
        )
    )
    .orderBy(
        F.col("delay_rate").desc()
    )
)

display(
    origin_airport_performance.limit(
        cfg.TOP_N_RESULTS
    )
)

#### Destination-airport performance

Destination airports are compared by operated-flight volume and arrival-delay
rate. A minimum-flight threshold is applied to reduce misleading rankings
based on small samples.

In [0]:
destination_airport_performance = (
    df_operated
    .join(
        df_airports_destination,
        cfg.DESTINATION_COLUMN,
        "left"
    )
    .groupBy(
        cfg.DESTINATION_COLUMN,
        "destination_airport_name",
        cfg.DESTINATION_CITY_COLUMN
    )
    .agg(
        F.count("*").alias("operated_flights"),

        F.sum(
            F.when(
                F.col(cfg.ARRIVAL_DELAY_FLAG_COLUMN) == 1,
                1
            ).otherwise(0)
        ).alias("delayed_flights"),

        F.avg(
            cfg.ARRIVAL_DELAY_COLUMN
        ).alias("average_arrival_delay")
    )
    .filter(
        F.col("operated_flights")
        >= cfg.MIN_OPERATIONAL_FLIGHTS
    )
    .withColumn(
        "delay_rate",
        F.round(
            F.col("delayed_flights")
            / F.col("operated_flights")
            * 100,
            2
        )
    )
    .withColumn(
        "average_arrival_delay",
        F.round(
            F.col("average_arrival_delay"),
            2
        )
    )
    .orderBy(
        F.col("delay_rate").desc(),
        F.col("operated_flights").desc()
    )
)

display(
    destination_airport_performance.limit(
        cfg.TOP_N_RESULTS
    )
)

#### Scheduled departure-hour analysis

This section examines whether arrival-delay rates differ by scheduled
departure hour.

In [0]:
departure_hour_performance = (
    df_operated
    .withColumn(
        # CRS_DEP_TIME occasionally uses "2400" for a scheduled midnight
        # departure; floor(.../100) alone would yield an invalid hour of 24,
        # so the result is taken modulo 24 to normalize midnight to hour 0.
        "scheduled_departure_hour",
        (F.floor(
            F.col("CRS_DEP_TIME") / 100
        ) % 24).cast("int")
    )
    .groupBy("scheduled_departure_hour")
    .agg(
        F.count("*").alias("operated_flights"),

        F.sum(
            F.when(
                F.col(cfg.ARRIVAL_DELAY_FLAG_COLUMN) == 1,
                1
            ).otherwise(0)
        ).alias("delayed_flights"),

        F.avg(
            cfg.ARRIVAL_DELAY_COLUMN
        ).alias("average_arrival_delay")
    )
    .withColumn(
        "delay_rate",
        F.round(
            F.col("delayed_flights")
            / F.col("operated_flights")
            * 100,
            2
        )
    )
    .withColumn(
        "average_arrival_delay",
        F.round(
            F.col("average_arrival_delay"),
            2
        )
    )
    .orderBy("scheduled_departure_hour")
)

display(departure_hour_performance)

#### Visualization

The following line chart compares arrival delay rates across scheduled departure hours.

The results indicate relatively lower delay rates during the early morning hours, followed by a gradual increase throughout the day. Delay rates peak during the late afternoon and early evening before declining for the latest scheduled departures. This pattern suggests that operational congestion and accumulated delays may contribute to higher arrival delay rates later in the day.

In [0]:
departure_hour_plot = (
    departure_hour_performance
    .orderBy("scheduled_departure_hour")
    .toPandas()
)

plt.figure(figsize=(11, 5))

plt.plot(
    departure_hour_plot["scheduled_departure_hour"],
    departure_hour_plot["delay_rate"],
    marker="o"
)

plt.title("Arrival Delay Rate by Scheduled Departure Hour")
plt.xlabel("Scheduled Departure Hour")
plt.ylabel("Delay Rate (%)")
plt.xticks(range(0, 24))
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.show()

#### Time-of-day analysis

Scheduled departures are grouped into time-of-day periods to provide a more
interpretable operational comparison.

In [0]:
# Categorize scheduled departures using the project's standard time-of-day
# periods (cfg.TIME_OF_DAY_BUCKETS - the same definition used later for the
# TIME_OF_DAY model feature in feature engineering)

time_of_day_expression = None
time_of_day_order_expression = None

for bucket_order, bucket in enumerate(cfg.TIME_OF_DAY_BUCKETS):
    hour_condition = F.col("scheduled_departure_hour").between(
        bucket["start_hour"], bucket["end_hour"]
    )

    if time_of_day_expression is None:
        time_of_day_expression = F.when(hour_condition, bucket["label"])
        time_of_day_order_expression = F.when(hour_condition, bucket_order)
    else:
        time_of_day_expression = time_of_day_expression.when(
            hour_condition, bucket["label"]
        )
        time_of_day_order_expression = time_of_day_order_expression.when(
            hour_condition, bucket_order
        )

time_of_day_performance = (
    df_operated
    .withColumn(
        # CRS_DEP_TIME occasionally uses "2400" for a scheduled midnight
        # departure; floor(.../100) alone would yield an invalid hour of 24,
        # so the result is taken modulo 24 to normalize midnight to hour 0.
        "scheduled_departure_hour",
        (F.floor(
            F.col("CRS_DEP_TIME") / 100
        ) % 24).cast("int")
    )
    .withColumn(
        "time_of_day",
        time_of_day_expression
    )
    .withColumn(
        "time_of_day_order",
        time_of_day_order_expression
    )
    .groupBy(
        "time_of_day",
        "time_of_day_order"
    )
    .agg(
        F.count("*").alias("operated_flights"),

        F.sum(
            F.when(
                F.col(cfg.ARRIVAL_DELAY_FLAG_COLUMN) == 1,
                1
            ).otherwise(0)
        ).alias("delayed_flights"),

        F.avg(
            cfg.ARRIVAL_DELAY_COLUMN
        ).alias("average_arrival_delay")
    )
    .withColumn(
        "delay_rate",
        F.round(
            F.col("delayed_flights")
            / F.col("operated_flights")
            * 100,
            2
        )
    )
    .withColumn(
        "average_arrival_delay",
        F.round(
            F.col("average_arrival_delay"),
            2
        )
    )
    .orderBy("time_of_day_order")
)

display(
    time_of_day_performance.drop("time_of_day_order")
)

#### Visualization

The following bar chart compares arrival delay rates across the project's five standard time-of-day periods (Overnight, Morning, Afternoon, Evening, Night — matching `cfg.TIME_OF_DAY_BUCKETS`, the same definition used later for the `TIME_OF_DAY` model feature).

Delay rates are generally expected to be lowest during the overnight and morning periods and to increase later in the day as operational congestion and accumulated delays build up. The exact pattern — including where the separately reported Night period falls relative to Evening — should be confirmed against the output once this notebook is re-run with these corrected time-of-day categories.

In [0]:
time_of_day_plot = (
    time_of_day_performance
    .orderBy("time_of_day_order")
    .toPandas()
)

plt.figure(figsize=(8, 5))

plt.bar(
    time_of_day_plot["time_of_day"],
    time_of_day_plot["delay_rate"]
)

plt.title("Arrival Delay Rate by Time of Day")
plt.xlabel("Time of Day")
plt.ylabel("Delay Rate (%)")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.show()

#### Seasonal analysis

Months are grouped into meteorological seasons to identify broader seasonal
differences in flight-delay performance.

In [0]:
# Categorize months using the project's standard meteorological seasons
# (cfg.SEASON_MONTH_GROUPS - the same definition used later for the SEASON
# model feature in feature engineering)

season_expression = None
season_order_expression = None

for season_order, (season_name, season_months) in enumerate(
    cfg.SEASON_MONTH_GROUPS.items()
):
    month_condition = F.col(cfg.MONTH_COLUMN).isin(*season_months)

    if season_expression is None:
        season_expression = F.when(month_condition, season_name)
        season_order_expression = F.when(month_condition, season_order)
    else:
        season_expression = season_expression.when(
            month_condition, season_name
        )
        season_order_expression = season_order_expression.when(
            month_condition, season_order
        )

seasonal_performance = (
    df_operated
    .withColumn("season", season_expression)
    .withColumn("season_order", season_order_expression)
    .groupBy(
        "season",
        "season_order"
    )
    .agg(
        F.count("*").alias("operated_flights"),

        F.sum(
            F.when(
                F.col(cfg.ARRIVAL_DELAY_FLAG_COLUMN) == 1,
                1
            ).otherwise(0)
        ).alias("delayed_flights"),

        F.avg(
            cfg.ARRIVAL_DELAY_COLUMN
        ).alias("average_arrival_delay")
    )
    .withColumn(
        "delay_rate",
        F.round(
            F.col("delayed_flights")
            / F.col("operated_flights")
            * 100,
            2
        )
    )
    .withColumn(
        "average_arrival_delay",
        F.round(
            F.col("average_arrival_delay"),
            2
        )
    )
    .orderBy("season_order")
)

display(
    seasonal_performance.drop("season_order")
)

#### Visualization

The following chart compares arrival delay rates across meteorological seasons, helping identify broader seasonal patterns beyond individual months.

In [0]:
seasonal_plot = (
    seasonal_performance
    .orderBy("season_order")
    .toPandas()
)

plt.figure(figsize=(8, 5))

plt.bar(
    seasonal_plot["season"],
    seasonal_plot["delay_rate"]
)

plt.title("Arrival Delay Rate by Season")
plt.xlabel("Season")
plt.ylabel("Delay Rate (%)")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.show()

#### Route performance

Routes are compared using their operated-flight volume and arrival-delay rate. A minimum-flight threshold is applied to avoid ranking routes with insufficient observations.

In [0]:
route_performance = (
    df_operated
    .join(
        df_airports_origin,
        cfg.ORIGIN_COLUMN,
        "left"
    )
    .join(
        df_airports_destination,
        cfg.DESTINATION_COLUMN,
        "left"
    )
    .groupBy(
        cfg.ORIGIN_COLUMN,
        "origin_airport_name",
        cfg.DESTINATION_COLUMN,
        "destination_airport_name"
    )
    .agg(
        F.count("*").alias("operated_flights"),

        F.sum(
            F.when(
                F.col(cfg.ARRIVAL_DELAY_FLAG_COLUMN) == 1,
                1
            ).otherwise(0)
        ).alias("delayed_flights")
    )
    .filter(
        F.col("operated_flights")
        >= cfg.MIN_ROUTE_FLIGHTS
    )
    .withColumn(
        "delay_rate",
        F.round(
            F.col("delayed_flights")
            / F.col("operated_flights")
            * 100,
            2
        )
    )
    .orderBy(
        F.col("delay_rate").desc(),
        F.col("operated_flights").desc()
    )
)

display(
    route_performance.limit(cfg.TOP_N_RESULTS)
)

#### Delay causes

This section measures the total delay minutes attributed to each reported cause. These variables are populated primarily for flights with significant arrival delays.

In [0]:
delay_cause_result = (
    df_operated
    .select(
        [
            F.sum(
                F.coalesce(
                    F.col(column_name),
                    F.lit(0)
                )
            ).alias(column_name)
            for column_name in cfg.DELAY_CAUSE_COLUMNS
        ]
    )
    .first()
)

delay_causes = (
    spark.createDataFrame(
        [
            (
                column_name,
                float(delay_cause_result[column_name] or 0)
            )
            for column_name in cfg.DELAY_CAUSE_COLUMNS
        ],
        [
            "delay_cause",
            "total_delay_minutes"
        ]
    )
    .orderBy(
        F.col("total_delay_minutes").desc()
    )
)

display(delay_causes)

#### Visualization

The following chart compares the total delay minutes attributed to each reported cause. This helps identify the operational factors that contribute most to flight delays.

In [0]:
delay_causes_plot = (
    delay_causes
    .orderBy(F.col("total_delay_minutes").asc())
    .toPandas()
)

delay_causes_plot["delay_cause"] = (
    delay_causes_plot["delay_cause"]
    .str.replace("_DELAY", "", regex=False)
    .str.replace("_", " ", regex=False)
    .str.title()
    .replace({
        "Nas": "National Airspace System"
    })
)

delay_causes_plot["total_delay_millions"] = (
    delay_causes_plot["total_delay_minutes"] / 1_000_000
)

plt.figure(figsize=(10, 6))

bars = plt.barh(
    delay_causes_plot["delay_cause"],
    delay_causes_plot["total_delay_millions"]
)

plt.title("Total Delay Minutes by Cause")
plt.xlabel("Total Delay Minutes (Millions)")
plt.ylabel("Delay Cause")
plt.grid(axis="x", alpha=0.3)

for bar in bars:
    value = bar.get_width()

    plt.text(
        value,
        bar.get_y() + bar.get_height() / 2,
        f" {value:.1f}M",
        va="center"
    )

plt.tight_layout()
plt.show()

#### Cancellation reasons

This section examines the distribution of cancellation reasons using the descriptive values provided by the cancellation-code reference table.

In [0]:
cancellation_reasons = (
    df
    .filter(
        F.col(cfg.CANCELLED_COLUMN) == 1
    )
    .groupBy(
        cfg.CANCELLATION_CODE_COLUMN
    )
    .count()
    .join(
        df_cancellation_codes,
        cfg.CANCELLATION_CODE_COLUMN,
        "left"
    )
    .select(
        cfg.CANCELLATION_CODE_COLUMN,
        "cancellation_reason",
        F.col("count").alias("cancelled_flights")
    )
    .orderBy(
        F.col("cancelled_flights").desc()
    )
)

display(cancellation_reasons)

#### Visualization

The following chart shows the distribution of flight cancellations by reason, highlighting the most common causes of cancelled flights.

In [0]:
cancellation_plot = (
    cancellation_reasons
    .orderBy(F.col("cancelled_flights").asc())
    .toPandas()
)

plt.figure(figsize=(9, 5))

bars = plt.barh(
    cancellation_plot["cancellation_reason"],
    cancellation_plot["cancelled_flights"]
)

plt.title("Cancelled Flights by Reason")
plt.xlabel("Number of Cancelled Flights")
plt.ylabel("Cancellation Reason")
plt.grid(axis="x", alpha=0.3)

for bar in bars:
    value = int(bar.get_width())

    plt.text(
        value,
        bar.get_y() + bar.get_height() / 2,
        f" {value:,}",
        va="center"
    )

plt.tight_layout()
plt.show()

#### Descriptive analytics completion

This descriptive analytics stage summarized the main temporal, airline, airport, route, delay-cause, and cancellation patterns in the dataset. These findings will guide the diagnostic-analysis and feature-engineering decisions in the following notebooks.

In [0]:
print("Descriptive analytics completed successfully.")